# 1. Importing necessary libraries and reading clean dataset

In [28]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import date
from pandas.tseries.holiday import AbstractHolidayCalendar, Holiday, nearest_workday, EasterMonday, GoodFriday

%matplotlib inline

import warnings
warnings.filterwarnings("ignore")

In [29]:
#Reading the Data

df = pd.read_csv('/Users/arthurassis/Documents/CCT - Data Analytics for Business/Machine Learning/Semester 2/CA01-MachineLearning-Semester2/CA01-MachineLearning/Python-Forecasting-Electrical-Demand-on-Spain/part_1_output_clean_data.csv',
                parse_dates= ['timestamp'])

In [30]:
df.head()

,timestamp,total load actual,total load forecast
0,2015-01-01 00:00:00,25385.0,26118.0
1,2015-01-01 01:00:00,24382.0,24934.0
2,2015-01-01 02:00:00,22734.0,23515.0
3,2015-01-01 03:00:00,21286.0,22642.0
4,2015-01-01 04:00:00,20264.0,21785.0


# 4. Time Feature Engineering

To easen the visualisation of the data, timestamp will be split into separates features that represent the periods of time. That allow grouping data by specific time windows and visualising patterns within it. The features that will be added are:

1. Year
2. Month
3. Day of Year (doy)
4. Day of Month (doy)
5. Day of week (dow)
6. Week of year (woy)
7. Hour


In [31]:
#Adding time features to our dataset

df['year'] = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month
df['doy'] = df['timestamp'].dt.dayofyear
df['dom'] = df['timestamp'].dt.day
df['dow'] = df['timestamp'].dt.weekday
df['woy'] = df['timestamp'].dt.isocalendar()['week']
df['hour'] = df['timestamp'].dt.hour
df['is_leap_year'] = df['timestamp'].dt.is_leap_year

Through the development of the project, another features had to be derived to improve models efficiency, due to the nature of the data. 

Cyclical variables such as month and doy were decomposed using sin and cosine, so that models are provided with a feature that represents its cyclical behavior. The way the dataset currently is, the doy 365 is not understood as being close from doy 1, that is what the cyclical fixes. When variables are transformed using cyclical functions such as sin and cosine, ML models can easily identify that the end of one year is close to the beginning of the next, and that the end of a week is close to the beginning of the next. That helps in convergency and overall in accuracy. 

It was also verifyied that holidays and the end of the year represent anomalous behaviour compared to the regular weeks, changing substantialy the target variable, therefore, feature identifying those events will be added, so that the model is able to identify them. 

Also weekends present a different level of load compared to weekdays, and to help the models to identify this difference, weekends were explicitly identified in the dataset.

In [32]:
#Defining a function that return the frequency of a certain feature in a dataset
def frequency(df):
    unique = np.sort(df.unique())      #Find unique values in a df
    steps = np.diff(unique)            #Verify the steps between this unique values
    freq = pd.Series(steps).unique()   #Verify the unique values for the step
    if len(freq) == 1:
        return freq
    else: return 'Irregular frequency'

In [33]:
#Verifying the unique values of the cyclical features
print('Month Min:', df['month'].min(), 
      'Max:', df['month'].max(), 
      'and frequency:',  frequency(df['month']))
print('doy Min:', df['doy'].min(), 
      'Max:', df['doy'].max(), 
      'and frequency:',  frequency(df['doy']))
print('dom Min:', df['dom'].min(), 
      'Max:', df['dom'].max(), 
      'and frequency:',  frequency(df['dom']))
print('dow Min:', df['dow'].min(), 
      'Max:', df['dow'].max(), 
      'and frequency:',  frequency(df['dow']))
print('woy Min:', df['woy'].min(), 
      'Max:', df['woy'].max(), 
      'and frequency:',  frequency(df['woy']))
print('Hour Min:', df['hour'].min(), 
      'Max:', df['hour'].max(), 
      'and frequency:',  frequency(df['hour']))

Month Min: 1 Max: 12 and frequency: [1]
doy Min: 1 Max: 366 and frequency: [1]
dom Min: 1 Max: 31 and frequency: [1]
dow Min: 0 Max: 6 and frequency: [1]
woy Min: 1 Max: 53 and frequency: [1]
Hour Min: 0 Max: 23 and frequency: [1]


In [34]:
#Adding cyclical features to the dataset

df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)
df['dow_sin'] = np.sin(2*np.pi*df['dow']/7)
df['dow_cos'] = np.cos(2*np.pi*df['dow']/7)
df['month_sin'] = np.sin(2*np.pi*df['month']/12)
df['month_cos'] = np.cos(2*np.pi*df['month']/12)
#The numpy function where is used to identify leap years and create the right cycle for doy and woy
df['doy_sin'] = np.where(df['is_leap_year'], np.sin(2*np.pi*df['doy']/366), np.sin(2*np.pi*df['doy']/365))
df['doy_cos'] = np.where(df['is_leap_year'], np.cos(2*np.pi*df['doy']/366), np.cos(2*np.pi*df['doy']/365))
df['woy_sin'] = np.where(df['is_leap_year'], np.sin(2*np.pi*df['woy']/53), np.sin(2*np.pi*df['woy']/52))
df['woy_cos'] = np.where(df['is_leap_year'], np.cos(2*np.pi*df['woy']/53), np.cos(2*np.pi*df['woy']/52)) 

In [35]:
#Adding a seasonal category to the dataset

def month_to_season(m):
    if m in [12,1,2]:
        return "Winter"
    elif m in [3,4,5]:
        return "Spring"
    elif m in [6,7,8]:
        return "Summer"
    else:   #9,10,11
        return "Autumn"
        
df['season'] = df['month'].map(month_to_season)

In [36]:
#Adding a feature do identify weekends using dow

df['is_weekend'] = df['dow'] > 4

In [37]:
#Adding a feature to identify the end of the year using doy

df['is_end_of_year'] = df['doy'] > 358

In [38]:
#Adding a feature to identify Christmas season

df['is_christmas'] = (df['month'] == 12) & (df['dom'] > 23) &  (df['dom'] < 27) 

In [39]:
#Collecting the Holidays of Spain

class SpainHolidayCalendar(AbstractHolidayCalendar):
    rules = [
        # Fixed national holidays
        Holiday('New Years Day', month=1, day=1),
        Holiday('Epiphany', month=1, day=6),
        GoodFriday,                           # Easter Friday (computed)
        Holiday('Labor Day', month=5, day=1),
        Holiday('Assumption', month=8, day=15),
        Holiday('Hispanic Heritage', month=10, day=12),
        Holiday('All Saints', month=11, day=1),
        Holiday('Constitution Day', month=12, day=6),
        Holiday('Immaculate Conception', month=12, day=8),
        Holiday('Christmas', month=12, day=25),
        
        # Easter Monday (national/regional)
        EasterMonday,
    ]

# Generate 2015-2018
cal = SpainHolidayCalendar()
holidays_2015_2018 = cal.holidays(start='2015-01-01', end='2018-12-31')
df_holidays = pd.DataFrame({'date': holidays_2015_2018})
df_holidays['is_holiday'] = 1

print("Spain holidays 2015-2018:", len(df_holidays))
print(df_holidays.head(20))

Spain holidays 2015-2018: 44
         date  is_holiday
0  2015-01-01           1
1  2015-01-06           1
2  2015-04-03           1
3  2015-04-06           1
4  2015-05-01           1
5  2015-08-15           1
6  2015-10-12           1
7  2015-11-01           1
8  2015-12-06           1
9  2015-12-08           1
10 2015-12-25           1
11 2016-01-01           1
12 2016-01-06           1
13 2016-03-25           1
14 2016-03-28           1
15 2016-05-01           1
16 2016-08-15           1
17 2016-10-12           1
18 2016-11-01           1
19 2016-12-06           1


In [40]:
# Merge holidays with my dataset
df['date'] = pd.to_datetime(df['timestamp'].dt.date)
df = df.merge(df_holidays, on='date', how='left')
df['is_holiday'] = df['is_holiday'].fillna(0).astype(int)
df.drop(columns = 'date', inplace = True)

In [41]:
#Visualising the dataframe after the alterations

df.head()

,timestamp,total load actual,total load forecast,year,month,doy,dom,dow,woy,hour,...,month_cos,doy_sin,doy_cos,woy_sin,woy_cos,season,is_weekend,is_end_of_year,is_christmas,is_holiday
0,2015-01-01 00:00:00,25385.0,26118.0,2015,1,1,1,3,1,0,...,0.866025,0.017213,0.999852,0.120537,0.992709,Winter,False,False,False,1
1,2015-01-01 01:00:00,24382.0,24934.0,2015,1,1,1,3,1,1,...,0.866025,0.017213,0.999852,0.120537,0.992709,Winter,False,False,False,1
2,2015-01-01 02:00:00,22734.0,23515.0,2015,1,1,1,3,1,2,...,0.866025,0.017213,0.999852,0.120537,0.992709,Winter,False,False,False,1
3,2015-01-01 03:00:00,21286.0,22642.0,2015,1,1,1,3,1,3,...,0.866025,0.017213,0.999852,0.120537,0.992709,Winter,False,False,False,1
4,2015-01-01 04:00:00,20264.0,21785.0,2015,1,1,1,3,1,4,...,0.866025,0.017213,0.999852,0.120537,0.992709,Winter,False,False,False,1


In [43]:
#Printing information of all the columns in the dataset

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35064 entries, 0 to 35063
Data columns (total 26 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   timestamp            35064 non-null  datetime64[ns]
 1   total load actual    35064 non-null  float64       
 2   total load forecast  35064 non-null  float64       
 3   year                 35064 non-null  int32         
 4   month                35064 non-null  int32         
 5   doy                  35064 non-null  int32         
 6   dom                  35064 non-null  int32         
 7   dow                  35064 non-null  int32         
 8   woy                  35064 non-null  UInt32        
 9   hour                 35064 non-null  int32         
 10  is_leap_year         35064 non-null  bool          
 11  hour_sin             35064 non-null  float64       
 12  hour_cos             35064 non-null  float64       
 13  dow_sin              35064 non-

In [46]:
#Exporting the file with with all the features

df.to_csv('part_2_output_features_engineered.csv', index = False)